# Задача 01. Динамика выручки и среднего чека по месяцам

**Постановка задачи:** руководитель маркетплейса хочет посмотреть, как менялись продажи по месяцам. Нужно посчитать только по доставленным заказам:

- количество заказов;
- количество покупателей;
- выручку;
- валовую прибыль;
- средний чек.

Результат отсортировать по месяцу. Используем SQL и базу `marketplace.sqlite`.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH order_finance AS (
    SELECT
        o.order_id,
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue,
        SUM(oi.quantity * (oi.unit_price * (1 - oi.discount_pct / 100.0) - p.cost)) AS gross_profit
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id, month
)
SELECT
    month,
    COUNT(order_id) AS delivered_orders,
    COUNT(DISTINCT customer_id) AS buyers,
    ROUND(SUM(revenue), 2) AS revenue,
    ROUND(SUM(gross_profit), 2) AS gross_profit,
    ROUND(SUM(revenue) / COUNT(order_id), 2) AS avg_order_value
FROM order_finance
GROUP BY month
ORDER BY month;
"""

df = pd.read_sql_query(query, conn)
df

In [ ]:
# Небольшая проверка результата: месяц должен быть уникальным
assert df['month'].is_unique
print('Месяцев в отчёте:', len(df))

**Комментарий стажёра:** сначала собрал выручку на уровне заказа, а затем агрегировал по месяцу. Так средний чек считается корректно: выручка делится на количество заказов, а не на количество товарных строк.